# Advanced Visualization and Hierarchical Indexing

## Exercise 1: Bar Graph — Retail Sales by Product Category

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = ['Electronics', 'Clothing', 'Food & Beverage', 'Home & Garden',
               'Sports', 'Toys', 'Books', 'Automotive']
sales      = [142500, 98300, 76800, 65400, 54200, 43100, 31600, 27900]

# Sort descending for a ranked view
order  = np.argsort(sales)[::-1]
categories = [categories[i] for i in order]
sales      = [sales[i]      for i in order]

colors = plt.cm.Blues(np.linspace(0.45, 0.90, len(categories)))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(categories, sales, color=colors, edgecolor='white', width=0.65)

ax.set_xlabel('Product Category', fontsize=12, labelpad=8)
ax.set_ylabel('Total Sales ($)', fontsize=12)
ax.set_title('Annual Sales by Product Category — Retail Store', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_ylim(0, max(sales) * 1.14)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + max(sales) * 0.01,
            f'${h:,.0f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()

## Exercise 2: Hierarchical Indexing — Temperature Filter for Canadian Cities

In [ ]:
import pandas as pd
import numpy as np

# Build a multi-level dataset covering several countries and cities
np.random.seed(0)
dates = pd.date_range('2023-06-01', periods=30)

records = []
for country, cities, temp_range in [
    ('Canada',         ['Toronto', 'Vancouver', 'Montreal'], (18, 38)),
    ('United States',  ['New York', 'Chicago',  'Miami'],    (20, 40)),
    ('United Kingdom', ['London',   'Manchester'],           (10, 28)),
]:
    for city in cities:
        for date in dates:
            temp = np.random.uniform(*temp_range)
            records.append((country, city, date, round(temp, 1)))

df_temp = pd.DataFrame(records, columns=['Country', 'City', 'Date', 'Temperature_C'])
df_temp = df_temp.set_index(['Country', 'City', 'Date']).sort_index()

print("DataFrame shape  :", df_temp.shape)
print("Index levels     :", df_temp.index.names)
print("\nSample rows:")
df_temp.head(6)

In [ ]:
# --- Filter: Canada only, temperature > 30°C ---
canada_data = df_temp.loc['Canada']
hot_days_canada = canada_data[canada_data['Temperature_C'] > 30]

print(f"Days above 30°C in Canada: {len(hot_days_canada)}")
hot_days_canada

In [ ]:
# --- Count of hot days per Canadian city ---
print("Hot days (>30°C) per Canadian city:")
print(hot_days_canada.groupby(level='City').size().rename('Hot Days'))

# --- Cross-country comparison: mean temperature above 30°C ---
print("\nMean temperature on hot days (>30°C) per country:")
print(df_temp[df_temp['Temperature_C'] > 30]
      .groupby(level='Country')['Temperature_C'].mean().round(2))

### How hierarchical indexing simplifies multi-level data analysis

A **hierarchical (MultiIndex)** index allows a DataFrame to be organised along multiple dimensions simultaneously — here Country, City, and Date — without storing those identifiers as ordinary columns.

Key advantages:
- **Intuitive slicing**: `df.loc['Canada']` instantly restricts the entire frame to Canadian rows. Adding a second level (`df.loc[('Canada', 'Toronto')]`) drills down further without any boolean mask.
- **Grouped aggregations**: `groupby(level='City')` operates directly on the index, avoiding an extra `groupby` column reference.
- **Memory efficiency**: the index is stored once rather than repeated in every row of a column.
- **Readable code**: the nesting mirrors the conceptual structure of the data (world → country → city → date), making queries self-documenting.

## Exercise 3: Advanced Filtering with Hierarchical Indices — Salary above 50 000 per Department

In [ ]:
import pandas as pd

# DataFrame referenced in XP Exercise 4: employees with Department / Employee ID index
data = {
    'Department':  ['Engineering', 'Engineering', 'Engineering', 'Engineering',
                    'Marketing',   'Marketing',   'Marketing',
                    'HR',          'HR',          'HR',
                    'Finance',     'Finance',     'Finance', 'Finance'],
    'Employee_ID': ['E001', 'E002', 'E003', 'E004',
                    'M001', 'M002', 'M003',
                    'H001', 'H002', 'H003',
                    'F001', 'F002', 'F003', 'F004'],
    'Name':        ['Alice', 'Bob', 'Carol', 'David',
                    'Eva',   'Frank', 'Grace',
                    'Hank',  'Iris', 'Jack',
                    'Kate',  'Liam', 'Mia', 'Noah'],
    'Salary':      [92000, 45000, 67000, 55000,
                    48000, 72000, 39000,
                    61000, 44000, 58000,
                    85000, 49000, 53000, 77000],
    'Years_Exp':   [7, 2, 5, 3, 3, 6, 1, 4, 2, 5, 8, 3, 4, 6]
}

df_emp = pd.DataFrame(data).set_index(['Department', 'Employee_ID'])
print("Full employee DataFrame:")
df_emp

In [ ]:
# --- Filter: Salary > 50 000 within each Department ---
high_salary = df_emp[df_emp['Salary'] > 50_000]

print(f"Employees earning above $50,000 ({len(high_salary)} of {len(df_emp)}):")
high_salary

In [ ]:
# --- Count per department and department-level summary ---
print("Count of high-earners per department:")
print(high_salary.groupby(level='Department').size().rename('Employees > $50k'))

print("\nMean salary of high-earners per department:")
print(high_salary.groupby(level='Department')['Salary'].mean().round(0).rename('Avg Salary'))

print("\nDirect department-level access (Engineering only):")
print(df_emp.loc['Engineering'][df_emp.loc['Engineering']['Salary'] > 50_000])

**How hierarchical indexing helps here:**  
Using `df_emp.loc['Engineering']` immediately restricts the view to a single department without a boolean column filter. This makes department-scoped queries both faster and more readable, especially as the number of departments or nesting levels grows.

## Exercise 4: Distribution Plot — MCU Movie Durations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---- MCU dataset (all theatrical releases through 2023) ----
mcu_data = [
    ('Iron Man',                       2008, 1, 126, 585.2,  98.6,  94, 91,  318.3,  585.2),
    ('The Incredible Hulk',            2008, 1, 114, 150.0, 132.5,  67, 71,  134.8,  263.4),
    ('Iron Man 2',                     2010, 1, 124, 200.0, 128.1,  73, 72,  312.4,  623.9),
    ('Thor',                           2011, 1, 115, 150.0,  65.7,  77, 76,  181.0,  449.3),
    ('Captain America: First Av.',     2011, 1, 124, 140.0,  65.1,  80, 74,  176.7,  370.6),
    ('The Avengers',                   2012, 1, 143, 220.0, 207.4,  92, 91,  623.4, 1519.6),
    ('Iron Man 3',                     2013, 2, 130, 200.0, 174.1,  79, 78,  409.0, 1214.8),
    ('Thor: The Dark World',           2013, 2, 112, 170.0,  85.7,  66, 76,  206.4,  644.8),
    ('Captain America: Winter Soldier',2014, 2, 136, 170.0,  95.0,  89, 92,  259.8,  714.4),
    ('Guardians of the Galaxy',        2014, 2, 121, 170.0,  94.3,  91, 92,  333.2,  773.3),
    ('Avengers: Age of Ultron',        2015, 2, 141, 250.0, 191.3,  75, 83,  459.0, 1402.8),
    ('Ant-Man',                        2015, 2, 117, 130.0,  57.2,  82, 86,  180.2,  519.3),
    ('Captain America: Civil War',     2016, 3, 147, 250.0, 179.1,  91, 89,  408.1, 1153.3),
    ('Doctor Strange',                 2016, 3, 115, 165.0,  85.1,  89, 86,  232.6,  677.7),
    ('Guardians of the Galaxy Vol. 2', 2017, 3, 136, 200.0, 146.5,  84, 87,  389.8,  863.8),
    ('Spider-Man: Homecoming',         2017, 3, 133, 175.0, 117.0,  92, 87,  380.1,  880.2),
    ('Thor: Ragnarok',                 2017, 3, 130, 180.0, 427.0,  93, 87,  315.1,  853.9),
    ('Black Panther',                  2018, 3, 134, 200.0, 242.1,  96, 79,  700.1, 1346.9),
    ('Avengers: Infinity War',         2018, 3, 149, 325.0, 257.7,  85, 91,  678.8, 2048.4),
    ('Ant-Man and the Wasp',           2018, 3, 118, 130.0,  75.8,  87, 91,  216.6,  622.7),
    ('Captain Marvel',                 2019, 3, 124, 175.0, 153.4,  79, 45,  426.8, 1128.3),
    ('Avengers: Endgame',              2019, 3, 181, 356.0, 357.1,  94, 90,  858.4, 2797.8),
    ('Spider-Man: Far From Home',      2019, 3, 129, 160.0, 185.1,  91, 95,  390.5, 1131.9),
    ('Black Widow',                    2021, 4, 134, 200.0,  80.0,  79, 91,  183.7,  379.8),
    ('Shang-Chi',                      2021, 4, 132, 150.0,  75.0,  91, 98,  224.5,  432.2),
    ('Eternals',                       2021, 4, 157, 200.0,  71.3,  47, 78,  164.9,  402.1),
    ('Spider-Man: No Way Home',        2021, 4, 148, 200.0, 260.1,  93, 98,  804.8, 1901.4),
    ('Doctor Strange in the MOM',      2022, 4, 126, 200.0, 187.4,  74, 86,  411.3,  955.8),
    ('Thor: Love and Thunder',         2022, 4, 119, 250.0, 144.2,  66, 78,  343.3,  760.9),
    ('Black Panther: Wakanda Forever', 2022, 4, 161, 250.0, 181.3,  84, 94,  453.8,  859.2),
    ('Ant-Man and the Wasp: Quantumania',2023,5,124, 200.0, 106.1,  46, 83,  214.5,  476.1),
    ('Guardians of the Galaxy Vol. 3', 2023, 5, 150, 250.0, 118.4,  82, 94,  358.9,  845.6),
    ('The Marvels',                    2023, 5, 105, 220.0,  46.1,  62, 84,   84.3,  206.1),
]

mcu_cols = ['Title','Year','Phase','Duration_min',
            'Production_Budget_M','Opening_Weekend_M',
            'Tomato_Meter','Audience_Score',
            'Domestic_BO_M','Worldwide_BO_M']

mcu = pd.DataFrame(mcu_data, columns=mcu_cols)
mcu['mcu_phase'] = 'Phase ' + mcu['Phase'].astype(str)

# String versions of financial columns (as they would appear in a scraped dataset)
for col in ['Production_Budget_M','Opening_Weekend_M','Domestic_BO_M','Worldwide_BO_M']:
    mcu[col + '_str'] = mcu[col].apply(lambda x: f'${x:.1f}M')

print(f"MCU dataset: {len(mcu)} movies, {mcu['Phase'].nunique()} phases")
mcu[['Title','Year','mcu_phase','Duration_min','Tomato_Meter','Audience_Score']].head(8)

In [ ]:
# --- Distribution plot of movie durations ---
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(data=mcu, x='Duration_min', bins=12, kde=True,
             color='steelblue', edgecolor='white',
             line_kws={'linewidth': 2.5}, ax=ax)

# Mark mean and median
mean_dur   = mcu['Duration_min'].mean()
median_dur = mcu['Duration_min'].median()
ax.axvline(mean_dur,   color='crimson',    linestyle='--', linewidth=1.8, label=f'Mean ({mean_dur:.0f} min)')
ax.axvline(median_dur, color='darkorange', linestyle='--', linewidth=1.8, label=f'Median ({median_dur:.0f} min)')

ax.set_title('Distribution of MCU Movie Durations', fontsize=14, fontweight='bold')
ax.set_xlabel('Duration (minutes)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Duration stats (minutes):")
print(mcu['Duration_min'].describe().round(1))

In [ ]:
# --- Duration distribution by Phase (KDE per phase) ---
fig, ax = plt.subplots(figsize=(10, 5))

palette = {'Phase 1':'#4C72B0','Phase 2':'#DD8452','Phase 3':'#55A868',
           'Phase 4':'#C44E52','Phase 5':'#8172B2'}

for phase, group in mcu.groupby('mcu_phase'):
    sns.kdeplot(data=group, x='Duration_min', label=phase,
                fill=True, alpha=0.20, linewidth=2,
                color=palette.get(phase), ax=ax)

ax.set_title('MCU Movie Duration Distribution by Phase (KDE)', fontsize=14, fontweight='bold')
ax.set_xlabel('Duration (minutes)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(title='Phase')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Exercise 5: Box Plot — Tomato Meter vs Audience Scores

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Melt to long format: one row per (movie, score_type, value)
scores_long = mcu.melt(
    id_vars=['Title', 'mcu_phase'],
    value_vars=['Tomato_Meter', 'Audience_Score'],
    var_name='Score_Type',
    value_name='Score'
)
scores_long['Score_Type'] = scores_long['Score_Type'].map({
    'Tomato_Meter':   'Tomato Meter',
    'Audience_Score': 'Audience Score'
})

print(f"Long-format shape: {scores_long.shape}")
scores_long.head(6)

In [ ]:
# --- Box plot: overall comparison ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Left: overall distribution
sns.boxplot(data=scores_long, x='Score_Type', y='Score',
            palette={'Tomato Meter': '#E84040', 'Audience Score': '#4C72B0'},
            width=0.45, linewidth=1.5, ax=axes[0])
sns.stripplot(data=scores_long, x='Score_Type', y='Score',
              color='black', alpha=0.4, size=4, jitter=True, ax=axes[0])
axes[0].set_title('Tomato Meter vs Audience Score\n(All MCU Films)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Score Type', fontsize=11)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_ylim(0, 110)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Right: by phase
sns.boxplot(data=scores_long, x='mcu_phase', y='Score',
            hue='Score_Type',
            palette={'Tomato Meter': '#E84040', 'Audience Score': '#4C72B0'},
            linewidth=1.2, ax=axes[1])
axes[1].set_title('Score Distribution by MCU Phase', fontsize=13, fontweight='bold')
axes[1].set_xlabel('MCU Phase', fontsize=11)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_ylim(0, 110)
axes[1].legend(title='Score Type', fontsize=9)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('MCU Movie Score Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Summary statistics per score type:")
print(scores_long.groupby('Score_Type')['Score'].describe().round(1))

**Observations:**
- The Tomato Meter shows higher spread (wider IQR) than the Audience Score, indicating critics diverge more than general audiences.
- Audience scores tend to cluster in the 80–95 range across all phases; critics are more volatile.
- Phase 4 and 5 films show a notable dip in Tomato Meter medians compared to Phase 3, while Audience Scores remain relatively stable.

## Exercise 6: Pair Plot — Financial Metrics by MCU Phase

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Convert string financial columns to numeric (simulating a real scraped dataset) ---
for col_str, col_num in [
    ('Production_Budget_M_str', 'Budget_M'),
    ('Opening_Weekend_M_str',   'Opening_M'),
    ('Domestic_BO_M_str',       'Domestic_M'),
    ('Worldwide_BO_M_str',      'Worldwide_M'),
]:
    mcu[col_num] = (
        mcu[col_str]
        .str.replace('$', '', regex=False)
        .str.replace('M', '', regex=False)
        .astype(float)
    )

fin_cols = ['Budget_M', 'Opening_M', 'Domestic_M', 'Worldwide_M']
print("Converted financial columns (head):")
mcu[['Title', 'mcu_phase'] + fin_cols].head(6)

In [ ]:
# --- Pair plot of financial metrics, hue by MCU phase ---
pp_data = mcu[fin_cols + ['mcu_phase']].copy()

# Rename for cleaner axis labels
pp_data = pp_data.rename(columns={
    'Budget_M':    'Prod. Budget ($M)',
    'Opening_M':   'Opening Weekend ($M)',
    'Domestic_M':  'Domestic BO ($M)',
    'Worldwide_M': 'Worldwide BO ($M)'
})

palette = {'Phase 1':'#4C72B0','Phase 2':'#DD8452','Phase 3':'#55A868',
           'Phase 4':'#C44E52','Phase 5':'#8172B2'}

g = sns.pairplot(
    pp_data,
    hue='mcu_phase',
    palette=palette,
    diag_kind='kde',
    plot_kws={'alpha': 0.7, 's': 60, 'edgecolor': 'white', 'linewidth': 0.5},
    diag_kws={'fill': True, 'alpha': 0.35, 'linewidth': 1.8},
    corner=False
)

g.figure.suptitle('Pairwise Relationships Between MCU Financial Metrics by Phase',
                  fontsize=14, fontweight='bold', y=1.01)

g.add_legend(title='MCU Phase', bbox_to_anchor=(1.02, 0.5), loc='center left')
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation matrix of financial metrics ---
corr = mcu[fin_cols].rename(columns={
    'Budget_M':'Budget','Opening_M':'Opening','Domestic_M':'Domestic','Worldwide_M':'Worldwide'
}).corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, square=True, ax=ax)
ax.set_title('Correlation Matrix — MCU Financial Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey financial insights per phase:")
print(mcu.groupby('mcu_phase')[fin_cols].mean().round(1).to_string())

### Pair Plot Observations

- **Opening Weekend vs Worldwide Box Office** shows the strongest positive correlation: films that open big also tend to earn big globally. This is consistent across all phases.
- **Production Budget vs Worldwide BO**: a moderate positive relationship exists, but it is not deterministic — some high-budget films underperform (e.g., *The Marvels*) while some mid-budget entries exceed expectations.
- **Phase 3 films** (green) dominate the upper-right of most scatter plots, reflecting the era of peak MCU commercial performance (*Endgame*, *Infinity War*, *Black Panther*).
- **Phase 5** (purple) clusters in the lower financial ranges, indicating that recent films have not matched the box-office scale of Phase 3.
- The KDE diagonals show that Worldwide Box Office has a right-skewed distribution, driven by a handful of record-breaking blockbusters pulling the tail.